<div>
<img src="img/RDM_course_2024_alpha_crop.png" height="150"/>
</div>

# add-raman-spectroscopy

## development notebook

---

In [1]:
import rich
from rich.pretty import Pretty
import uuid
from datetime import datetime

import matplotlib
matplotlib.use("qtagg")  # Preferred backend, alternative is TkAgg
# %matplotlib ipympl     # Not working properly with matplotlib buttons
from Ramacropy.Ramacropy import RamanSpectra

from modules.fairvibspec import FAIRVibSpec, Experiment, Measurement, VibSpecType, MeasurementTypes, Detection, Dataset, Series, UnitDefinition, SIFParameters, Analysis

In [2]:
def pretty_print(output):
    rich.print(Pretty(output, max_length=10))

### Initialisation of the RamanSpectra object with typical SIF file

Ramacropy initialises from a .csv or .sif file. The .sif file is the output from the Raman spectrometer. As the laser wavelength is not available from the .sif file, we need to provide it manually.

In [3]:
my_spec = RamanSpectra("./data/Simon/298K-ADY79-10s-10acc.sif", laser_wavelength=785.0)

The library uses open-source package `sif_parser` to read the .sif file and creates a few useful attributes like numpy arrays for the x and y data, including a copy of the raw data, as well as the metadata as an ordered dictionary.

In [ ]:
print(f"Raman shift:\n{my_spec.RamanShift}")      # the x axis of your data, contains the shift at each y point
print(f"Timestamp:\n{my_spec.TimeStamp}")         # the time data of each spectrum in the series
print(f"Spectral data:\n{my_spec.SpectralData}")  # the actual y data of your file
print(f"Raw data:\n{my_spec.RawData}")            # the same as spectral data, but doesn't change if you perform processing (only .pkl and .sif)
print(f"Metadata:\n{my_spec.SpectralInfo}")       # this is the metadata from andor see sif_parser for more info (only .pkl and .sif)
print(f"Path to file:\n{my_spec.directory}")      # original path to file
print(f"File name:\n{my_spec.filelab}")           # original file name

### Cropping the data to the region of interest

Weirdly, the y-axis is longer than the x-axis, which Ramacropy does not accept. For now, we just crop the y-axis to the same length as the x-axis in the region of interest. Have to confirm with Simon, though.

In [5]:
my_spec.SpectralData = my_spec.SpectralData[601:1625]
my_spec.RawData = my_spec.RawData[601:1625]

### Creating the now Raman-capable data model

__Of course, this will be done automatically and in the background in the future__

I allowed myself to rename the old `IRAnalysis` repository and root object of the data model to `FAIRVibSpec` to better reflect the scope of the package.

In [ ]:
data_model = FAIRVibSpec(
    datetime_created=str(datetime.now()),
    contributors=[
        "Simon Krause <s.krause@fkf.mpg.de>",
        "Torsten Giess <torsten.giess@ibtb.uni-stuttgart.de>"
    ]
)
pretty_print(data_model)

In [ ]:
my_experiment = Experiment(
    name=my_spec.filelab,
)
pretty_print(my_experiment)

In [ ]:
new_measurement = Measurement(
    id=str(uuid.uuid4()),
    vib_spec_type=VibSpecType.RAMAN.value,
    measurement_type=MeasurementTypes.SAMPLE.value,
    detection=Detection.INTENSITY.value,
)
pretty_print(new_measurement)


In [ ]:
new_dataset = Dataset(
    timestamp=str(datetime.fromtimestamp(my_spec.SpectralInfo["ExperimentTime"])),
    x_axis=Series(
        data_array=my_spec.RamanShift,
        unit=UnitDefinition(
            name=my_spec.SpectralInfo["FrameAxis"].decode("utf-8"),
        )
    ),
    y_axis=Series(
        data_array=my_spec.SpectralData,
        unit=UnitDefinition(
            name=my_spec.SpectralInfo["DataType"].decode("utf-8"),
        )
    )
)
pretty_print(new_dataset)

Up to this point, a lot is the same as before. However, we now have the option to add instrument parameters to the measurement object. Currently, this is only implemented for SIF files, but will be extended to include other file types (as needed) in the future.

In [ ]:
raman_metadata = SIFParameters(
    sif_version=my_spec.SpectralInfo["SifVersion"],
    experiment_time=my_spec.SpectralInfo["ExperimentTime"],
    detector_temperature=my_spec.SpectralInfo["DetectorTemperature"],
    exposure_time=my_spec.SpectralInfo["ExposureTime"],
    cycle_time=my_spec.SpectralInfo["CycleTime"],
    accumulated_cycle_time=my_spec.SpectralInfo["AccumulatedCycleTime"],
    accumulated_cycles=my_spec.SpectralInfo["AccumulatedCycles"],
    stack_cycle_time=my_spec.SpectralInfo["StackCycleTime"],
    pixel_readout_time=my_spec.SpectralInfo["PixelReadoutTime"],
    gain_dac=my_spec.SpectralInfo["GainDAC"],
    gate_width=my_spec.SpectralInfo["GateWidth"],
    grating_blaze=my_spec.SpectralInfo["GratingBlaze"],
    detector_type=my_spec.SpectralInfo["DetectorType"],
    detector_dimensions=my_spec.SpectralInfo["DetectorDimensions"],
    original_filename=my_spec.SpectralInfo["OriginalFilename"].decode("utf-8"),
    shutter_time=my_spec.SpectralInfo["ShutterTime"],
    spectrograph=my_spec.SpectralInfo["spectrograph"],
    gate_gain=my_spec.SpectralInfo["GateGain"],
    gate_delay=my_spec.SpectralInfo["GateDelay"],
    sif_calibration_version=my_spec.SpectralInfo["SifCalbVersion"],
    calibration_data=my_spec.SpectralInfo["Calibration_data"],
    raman_excitation_wavelength=my_spec.SpectralInfo["RamanExWavelength"],
    frame_axis=my_spec.SpectralInfo["FrameAxis"].decode("utf-8"),
    data_type=my_spec.SpectralInfo["DataType"].decode("utf-8"),
    image_axis=my_spec.SpectralInfo["ImageAxis"].decode("utf-8"),
    number_of_frames=my_spec.SpectralInfo["NumberOfFrames"],
    number_of_sub_images=my_spec.SpectralInfo["NumberOfSubImages"],
    total_length=my_spec.SpectralInfo["TotalLength"],
    image_length=my_spec.SpectralInfo["ImageLength"],
    xbin=my_spec.SpectralInfo["xbin"],
    ybin=my_spec.SpectralInfo["ybin"],
    timestamp_of_0=my_spec.SpectralInfo["timestamp_of_0"],
    size=my_spec.SpectralInfo["size"],
    tile=str(my_spec.SpectralInfo["tile"]),
    offset=my_spec.SpectralInfo["offset"],
)
pretty_print(raman_metadata)

In [ ]:
new_measurement.measurement_data = new_dataset
new_measurement.instrument_parameters = raman_metadata

my_experiment.measurements.append(new_measurement)

data_model.experiment = my_experiment

pretty_print(data_model)

### Let's see (part of) what Ramacropy can do

Sadly, the widget-style `ipympl` backend of matplotlib does not work with the interactive plots without complete refactoring specifically for Jupter Notebooks. So we have to rely on pop-up windows for the interactive plots based on either `QtAgg` or `TkAgg`.

In [12]:
my_spec.plot_few()

In [13]:
my_spec.baseline(interactive=True, debounce_interval=10)

### Adding the analysis/processing to the data model and saving it as JSON

__Support for the Raman Allotrope Simple Model will be added in the future__

In [ ]:
analysis = Analysis(
    sample_reference=new_measurement.id,
    corrected_data=Dataset(
        timestamp=str(datetime.now()),
        x_axis=Series(
            data_array=my_spec.RamanShift,
            unit=UnitDefinition(name=my_spec.SpectralInfo["FrameAxis"].decode("utf-8")),
        ),
        y_axis=Series(
            data_array=my_spec.SpectralData,
            unit=UnitDefinition(name=my_spec.SpectralInfo["DataType"].decode("utf-8")),
        ),
    )
)
pretty_print(analysis)

In [ ]:
data_model.experiment.analysis.append(analysis)
pretty_print(data_model)

In [ ]:
with open(f"./data/Simon/{my_spec.filelab}.json", "w") as f:
    f.write(
        data_model.model_dump_json(
            indent=2,
            exclude_none=True
        )
    )
    print("Success! 🎉")

---

### \~\~\~under heavy development\~\~\~

#### 1) Time-course data from Raman will be added

Currently no data to test this with...

In [ ]:
my_spec.plot_kinetic()

#### 2) Normalisation and integration will be added
__They are currently broken even in original code__

In [ ]:
my_spec.normalise(method="area",interactive = True)

In [ ]:
my_spec.integrate(interactive=True)

#### 3) Full integration with `FAIRVibSpec` will be added

- Data model overhaul
- Full Raman integration
- Real-world testing with
  1. Pyridine FTIR of silica materials from Johanna Bruckner
  2. FTIR and Raman of chitosan from Achim Weber
  3. FTIR and Raman of MOFs from Simon Krause
- IR refactoring
- Documentation update
- Package release